# PaddleOCR-VL 정확도 실험 — 보험약관 표를 어떻게 넣어야 정확한가 (Colab T4)

모델은 PaddleOCR-VL로 확정했다. 이제 **같은 페이지**에서 다음 네 축을 바꿔 가며 어느 조합이 약관 표를 가장 정확히 뽑는지 본다. 레포 코드는 아직 바꾸지 않는다. 결과가 나오면 이긴 조합을 파이프라인에 옮긴다.

| 축 | 비교 | 지금 파이프라인 |
|---|---|---|
| 입력 단위 | **A** 페이지 전체 / **C** 레이아웃 모델(PP-DocLayout)이 잡은 표 영역만 잘라 넣기 / **D** PaddleOCR 공식 파이프라인 | A |
| 해상도 | **A** 150 DPI / **B** 200 DPI | 150 |
| 모델 | **1.6**(최신, 기본) / 1.5(11번 셀에서 비교) | — |
| 변환기 | 가로 병합 셀 미반복 패치 | 반복 |

판정 지표: 괘선 표에서 PyMuPDF 셀 대비 **셀 일치율**, 행마다 열 수가 흔들리는 **ragged 비율**, 표 없는 쪽의 **본문 오탐**, 페이지당 **시간**과 **입력 토큰**.

전제: T4 런타임, Drive `bench/.pdfs/`에 `<보험사>_<상품>.pdf`. 지난 노트북이 만든 `bench/insurers-trial/`의 페이지 선택이 있으면 그대로 쓴다. 소요는 설치 15분, A·B 3분, C 5분, D 10분, 1.5 비교 5분.


In [ ]:
# 1. Drive · 경로
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os, sys, json, time, shutil, subprocess, hashlib, glob, re, base64, io
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
BENCH_DATA = '/content/drive/MyDrive/bench'
PDFS = Path(BENCH_DATA) / '.pdfs'
SEL_DIR = Path(BENCH_DATA) / 'insurers-trial'; SEL_DIR.mkdir(parents=True, exist_ok=True)
PNG_DIR = SEL_DIR / 'pages'; PNG_DIR.mkdir(exist_ok=True)
OUT = Path(BENCH_DATA) / 'paddle-accuracy'; OUT.mkdir(exist_ok=True)   # 이 실험의 결과
os.environ['BENCH_DATA'] = BENCH_DATA


In [ ]:
# 2. 코드 + 변환기 패치(가로 병합 미반복). 파서는 레포 것을 그대로 쓴다 — 파이프라인과 같은 해석
%cd /content
!rm -rf Policy-Chunker && git clone -q https://github.com/SM17-YoonDongJu/Policy-Chunker
%cd /content/Policy-Chunker
!pip install -q -e .

p = Path('insurance_chunker/extractor.py'); src = p.read_text(encoding='utf-8')
old1 = '''            for _ in range(n_col):
                out_row.append(text)
                if n_row > 1:
                    spans[col] = (text, n_row - 1)
                col += 1
'''
new1 = '''            for k in range(n_col):
                cell = text if k == 0 else ""  # 가로 병합은 첫 칸에만
                out_row.append(cell)
                if n_row > 1:
                    spans[col] = (cell, n_row - 1)
                col += 1
'''
old2 = '''        elif tag == "lcel":
            row.append(row[-1] if row else "")
'''
new2 = '''        elif tag == "lcel":
            row.append("")  # 왼쪽(가로) 병합: 반복하지 않는다
'''
if new1 in src:
    print("패치 이미 적용됨")
else:
    assert old1 in src and old2 in src, "extractor.py가 바뀌어 패치 대상이 없다"
    p.write_text(src.replace(old1, new1).replace(old2, new2), encoding='utf-8'); print("colspan 패치 적용")
from insurance_chunker import extractor
parse_vlm = extractor._parse_vlm_output      # HTML/OTSL/파이프 → 파이프 마크다운


In [ ]:
# 3. 문서 — <보험사>_<상품>.pdf
import pymupdf
DOCS = {}
for f in sorted(PDFS.glob('*_*.pdf')):
    insurer, _, product = f.stem.partition('_')
    DOCS[re.sub(r'[^\w-]+', '-', f.stem).strip('-')] = {'pdf': str(f), 'insurer': insurer, 'note': product}
assert DOCS, "Drive bench/.pdfs/ 에 <보험사>_<상품>.pdf 를 올린다"
for k, v in DOCS.items():
    print(f"  {k:<28} {v['insurer']:<8} {pymupdf.open(v['pdf']).page_count:>5}쪽")


In [ ]:
# 4. 문서마다 6쪽 (괘선 표 2 · gap 2 · 긴 표 1 · 본문 1). 지난 노트북 결과가 있으면 재사용
def pick_pages(pdf):
    doc = pymupdf.open(pdf); nums = list(range(1, doc.page_count + 1))
    sources, gap, _ = extractor.extract_cpu_tables(pdf, nums, use_vision=True)
    pym = sources['pymupdf']; by_len = sorted(pym, key=lambda p: -len(pym[p]))
    plain = [p for p in nums if p not in pym and p not in gap]
    chosen = {}
    def take(kind, cands, n):
        for p in cands:
            if sum(1 for k in chosen.values() if k == kind) >= n: break
            if p not in chosen: chosen[p] = kind
    take('long', by_len[:1], 1)
    take('ruled', [by_len[len(by_len) // 4], by_len[len(by_len) // 2]] if len(by_len) >= 4 else by_len, 2)
    take('gap', gap, 2)
    take('plain', plain[len(plain) // 2:], 1)
    take('ruled', by_len, 6 - len(chosen) + sum(1 for k in chosen.values() if k == 'ruled'))
    return dict(sorted(chosen.items())), {'pymupdf_pages': len(pym), 'gap_pages': len(gap), 'total': doc.page_count}

SEL = {}
for did, d in DOCS.items():
    trial, meta = SEL_DIR / f'{did}-trial.pdf', SEL_DIR / f'{did}-pages.json'
    if trial.exists() and meta.exists():
        SEL[did] = json.load(open(meta)); print(f"{did}: 캐시"); continue
    t0 = time.time(); pages, info = pick_pages(d['pdf'])
    doc = pymupdf.open(d['pdf']); doc.select([p - 1 for p in pages]); doc.save(trial); doc.close()
    for i, page in enumerate(pymupdf.open(trial), 1):
        page.get_pixmap(dpi=110).save(PNG_DIR / f'{did}-p{i:02d}.png')
    SEL[did] = {'pages': {str(i): {'orig': p, 'kind': k} for i, (p, k) in enumerate(pages.items(), 1)}, **info}
    json.dump(SEL[did], open(meta, 'w'), ensure_ascii=False, indent=1)
    print(f"{did}: {info['total']}쪽, 표 {info['pymupdf_pages']} / gap {info['gap_pages']} → {pages} ({time.time() - t0:.0f}s)")

# PyMuPDF 기준 표 (괘선 표 페이지의 정답 대용)
PYM = {}
for did in DOCS:
    doc = pymupdf.open(str(SEL_DIR / f'{did}-trial.pdf'))
    PYM[did] = {str(i): extractor.extract_pymupdf(page) for i, page in enumerate(doc, 1)}


In [ ]:
# 5. vLLM 설치 (5~10분). torchaudio는 교체된 torch와 CUDA 버전이 달라 지운다
!pip install -q uv
!uv pip install --system -q vllm --torch-backend=auto
!pip uninstall -y -q torchaudio
!python -c "import vllm, transformers, torch; print('vllm', vllm.__version__, '/ transformers', transformers.__version__, '/ torch', torch.__version__, torch.version.cuda)"


In [ ]:
# 6. Paddle 서빙 함수 · VLM 호출 함수. 최신 1.6(2026-05-28, v3.6.0)으로 돌린다
import requests
VLLM = {'proc': None, 'served': None}

def serve_paddle(model_id, served, port=8000, wait_min=15):
    stop_paddle()
    cmd = (f"vllm serve {model_id} --trust-remote-code --dtype half --max-model-len 8192 "
           f"--max-num-batched-tokens 16384 --no-enable-prefix-caching --mm-processor-cache-gb 0 "
           f"--gpu-memory-utilization 0.5 --served-model-name {served} --port {port} "
           f"--trust-request-mm-kwargs")   # 공식 파이프라인(D)이 요청마다 min/max_pixels를 보낸다 — 없으면 400
    VLLM['proc'] = subprocess.Popen(cmd.split(), stdout=open('/tmp/vllm.log', 'w'), stderr=subprocess.STDOUT,
                                    start_new_session=True)   # 세션 단위로 죽일 수 있게
    VLLM['served'] = served
    for _ in range(wait_min * 12):
        if VLLM['proc'].poll() is not None: break
        try:
            if requests.get(f'http://localhost:{port}/v1/models', timeout=2).ok:
                print(f"vLLM 준비: {served}"); return
        except Exception: pass
        time.sleep(5)
    print(open('/tmp/vllm.log').read()[-3000:]); raise SystemExit("vLLM이 뜨지 않았다")

def _vram_used():
    out = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                         capture_output=True, text=True).stdout.strip().splitlines()
    return int(out[0]) if out else 0

def stop_paddle():
    """vLLM은 엔진을 자식 프로세스로 띄운다 — 부모만 죽이면 자식이 VRAM을 쥔 채 남아 다음 기동이 OOM으로 실패한다."""
    import signal
    if VLLM['proc'] and VLLM['proc'].poll() is None:
        try: os.killpg(os.getpgid(VLLM['proc'].pid), signal.SIGKILL)
        except Exception: VLLM['proc'].kill()
        VLLM['proc'].wait()
    subprocess.run(['pkill', '-9', '-f', 'vllm serve']); subprocess.run(['pkill', '-9', '-f', 'VLLM::'])
    for _ in range(30):
        if _vram_used() < 1500: break
        time.sleep(2)

def vlm(img_b64, prompt='Table Recognition:', max_tokens=4096):
    """→ (원문, 파싱된 마크다운 or None, 초, prompt_tokens, finish_reason)"""
    t0 = time.perf_counter()
    r = requests.post('http://localhost:8000/v1/chat/completions', json={
        'model': VLLM['served'], 'temperature': 0, 'max_tokens': max_tokens,
        'messages': [{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{img_b64}'}},
            {'type': 'text', 'text': prompt}]}]}, timeout=(5, 300))
    r.raise_for_status(); body = r.json(); ch = body['choices'][0]
    out = (ch['message']['content'] or '').strip()
    return out, parse_vlm(out), round(time.perf_counter() - t0, 2), (body.get('usage') or {}).get('prompt_tokens'), ch.get('finish_reason')

def page_png(did, i, dpi):
    page = pymupdf.open(str(SEL_DIR / f'{did}-trial.pdf'))[int(i) - 1]
    return page.get_pixmap(dpi=dpi).tobytes('png')

b64 = lambda png: base64.b64encode(png).decode()

# 결과 저장소 — 방법/문서/페이지 키로 Drive에 누적. 다시 돌리면 있는 건 건너뛴다
RES_PATH = OUT / 'results.json'
RES = json.load(open(RES_PATH)) if RES_PATH.exists() else {}
def put(method, did, i, rec):
    RES.setdefault(method, {}).setdefault(did, {})[str(i)] = rec
    json.dump(RES, open(RES_PATH, 'w'), ensure_ascii=False)
def done(method, did, i): return str(i) in RES.get(method, {}).get(did, {})

serve_paddle('PaddlePaddle/PaddleOCR-VL-1.6', 'PaddleOCR-VL-1.6-0.9B')
out, mdt, sec, ptok, fin = vlm(b64(page_png(next(iter(DOCS)), 1, 150)))
print(f"스모크: {sec}s prompt_tok={ptok} finish={fin}\n{out[:300]}")


In [ ]:
# 7. 방법 A·B — 페이지 전체를 150 / 200 DPI로 (지금 파이프라인과 같은 방식)
for method, dpi in (('A-page150', 150), ('B-page200', 200)):
    for did in DOCS:
        for i in SEL[did]['pages']:
            if done(method, did, i): continue
            out, mdt, sec, ptok, fin = vlm(b64(page_png(did, i, dpi)))
            put(method, did, i, {'raw': out, 'md': mdt, 'sec': sec, 'prompt_tokens': ptok, 'finish': fin, 'n_tables': 1 if mdt else 0})
        print(f"{method} {did} 완료")
print({m: sum(len(v) for v in RES[m].values()) for m in RES})


In [ ]:
# 8. 방법 C — 레이아웃 모델(CPU)이 잡은 표 영역만 잘라 VLM에. 표가 없다고 하면 호출하지 않는다
#    paddleocr 설치는 vLLM 환경과 충돌할 수 있어 A·B를 저장한 뒤에 한다
!pip install -q paddlepaddle==3.2.1 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
!pip install -q "paddleocr[doc-parser]>=3.6.0"
from PIL import Image
from paddleocr import LayoutDetection
layout = None
for name in ('PP-DocLayoutV3', 'PP-DocLayoutV2', 'PP-DocLayout_plus-L'):
    try:
        layout = LayoutDetection(model_name=name, device='cpu'); LAYOUT_NAME = name; break
    except Exception as e:
        print(f"{name} 실패: {str(e)[:120]}")
assert layout, "레이아웃 모델을 못 올렸다"
print("레이아웃 모델:", LAYOUT_NAME)

def table_boxes(png, min_score=0.5):
    img = Image.open(io.BytesIO(png)).convert('RGB'); tmp = '/tmp/_layout.png'; img.save(tmp)
    res = next(iter(layout.predict(tmp, batch_size=1, layout_nms=True)))
    boxes = [b for b in res.json['res']['boxes'] if b['label'] == 'table' and b['score'] >= min_score]
    return img, [b['coordinate'] for b in boxes]

def crop_b64(img, box, margin=8):
    x0, y0, x1, y1 = box
    c = img.crop((max(0, int(x0) - margin), max(0, int(y0) - margin), min(img.width, int(x1) + margin), min(img.height, int(y1) + margin)))
    buf = io.BytesIO(); c.save(buf, format='PNG'); return base64.b64encode(buf.getvalue()).decode()

method = 'C-crop200'
for did in DOCS:
    for i in SEL[did]['pages']:
        if done(method, did, i): continue
        t0 = time.perf_counter(); img, boxes = table_boxes(page_png(did, i, 200)); lay_sec = round(time.perf_counter() - t0, 2)
        mds, raws, secs, toks, fins = [], [], [], [], []
        for box in boxes:
            out, mdt, sec, ptok, fin = vlm(crop_b64(img, box))
            raws.append(out); secs.append(sec); toks.append(ptok); fins.append(fin)
            if mdt: mds.append(mdt)
        put(method, did, i, {'raw': '\n\n'.join(raws), 'md': '\n\n'.join(mds) or None, 'sec': round(lay_sec + sum(secs), 2),
                             'layout_sec': lay_sec, 'prompt_tokens': max(toks) if toks else None,
                             'finish': 'length' if 'length' in fins else ('stop' if fins else None),
                             'n_boxes': len(boxes), 'n_tables': len(mds)})
    print(f"{method} {did} 완료 — 표 영역 수:", [RES[method][did][i]['n_boxes'] for i in SEL[did]['pages']])


In [ ]:
# 9. 방법 D — PaddleOCR 공식 파이프라인 (레이아웃 → 블록별 인식, VLM은 같은 vLLM 서버)
from paddleocr import PaddleOCRVL
pipe = PaddleOCRVL(vl_rec_backend='vllm-server', vl_rec_server_url='http://localhost:8000/v1',
                   vl_rec_api_model_name=VLLM['served'], vl_rec_model_name=VLLM['served'],
                   layout_detection_model_name=LAYOUT_NAME, device='cpu',
                   use_chart_recognition=False, use_seal_recognition=False)
method = 'D-pipeline'
for did in DOCS:
    for i in SEL[did]['pages']:
        if done(method, did, i): continue
        tmp = f'/tmp/_{did}-{i}.png'; open(tmp, 'wb').write(page_png(did, i, 200))
        t0 = time.perf_counter(); res = next(iter(pipe.predict(tmp))); sec = round(time.perf_counter() - t0, 2)
        blocks = res.json['res'].get('parsing_res_list', [])
        tables = [b['block_content'] for b in blocks if b.get('block_label') == 'table']
        mds = [m for m in (parse_vlm(t) for t in tables) if m]
        put(method, did, i, {'raw': '\n\n'.join(tables), 'md': '\n\n'.join(mds) or None, 'sec': sec, 'prompt_tokens': None, 'finish': None,
                             'n_boxes': len(tables), 'n_tables': len(mds), 'n_blocks': len(blocks),
                             'page_md': (res.markdown or {}).get('markdown_texts', '')[:20000]})   # 본문까지 — 스캔 페이지 복구 참고용
    print(f"{method} {did} 완료")


In [ ]:
# 10. 지표 요약 (보험사 × 방법) + 페이지별 나란히 마크다운
import pandas as pd
from IPython.display import display, Markdown, Image as IImage

def cells(mdt):
    out = set()
    for ln in (mdt or '').splitlines():
        if '|' in ln and not re.fullmatch(r'[\s|:-]+', ln):
            out |= {c.strip() for c in ln.strip().strip('|').split('|') if c.strip()}
    return out

def ragged(mdt):
    widths = [ln.count('|') for ln in (mdt or '').splitlines() if '|' in ln and not re.fullmatch(r'[\s|:-]+', ln)]
    if len(widths) < 2: return 0.0
    mode = max(set(widths), key=widths.count); return round(1 - widths.count(mode) / len(widths), 2)

def pct(vals, q):
    s = sorted(v for v in vals if v is not None); return s[min(len(s) - 1, int(len(s) * q))] if s else None

rows = []
for did, d in DOCS.items():
    for m in RES:
        recs = RES[m].get(did, {});
        if not recs: continue
        sel = SEL[did]['pages']
        tbl = [p for p in sel if sel[p]['kind'] != 'plain']
        match = [len(cells(recs[p]['md']) & cells(PYM[did][p])) / len(cells(PYM[did][p]))
                 for p in sel if sel[p]['kind'] in ('ruled', 'long') and p in recs and recs[p]['md'] and cells(PYM[did].get(p))]
        rows.append({'보험사': d['insurer'], '문서': did, '방법': m,
                     '표 낸 쪽': f"{sum(1 for p in tbl if recs.get(p, {}).get('md'))}/{len(tbl)}",
                     '셀일치': round(sum(match) / len(match), 2) if match else None,
                     'ragged': round(sum(ragged(recs[p]['md']) for p in tbl if p in recs) / max(1, len(tbl)), 2),
                     '본문오탐': sum(1 for p in sel if sel[p]['kind'] == 'plain' and recs.get(p, {}).get('md')),
                     'p50초': pct([recs[p]['sec'] for p in recs], 0.5),
                     '잘림': sum(1 for p in recs if recs[p].get('finish') == 'length'),
                     '입력tok p50': pct([recs[p].get('prompt_tokens') for p in recs], 0.5)})
df = pd.DataFrame(rows).sort_values(['보험사', '문서', '방법']); display(df)
df.to_csv(OUT / 'summary.csv', index=False)
print(df.groupby('방법')[['셀일치', 'ragged', '본문오탐', 'p50초']].mean(numeric_only=True).round(2))

lines = ['# PaddleOCR-VL 정확도 실험', '']
for did, d in DOCS.items():
    lines += [f'## {did} ({d["insurer"]})', '']
    for p, info in SEL[did]['pages'].items():
        lines += [f'### p{p} — 원본 p{info["orig"]} [{info["kind"]}]', '', f'![{did}-p{p}](../insurers-trial/pages/{did}-p{int(p):02d}.png)', '',
                  '#### PyMuPDF', '', PYM[did].get(p) or '(없음)', '']
        for m in RES:
            r = RES[m].get(did, {}).get(p)
            if r: lines += [f'#### {m}  ({r["sec"]}s, 표 {r.get("n_tables")}, ragged {ragged(r["md"])})', '', r['md'] or '(출력 없음)', '']
(OUT / 'compare.md').write_text('\n'.join(lines), encoding='utf-8'); print("저장:", OUT / 'compare.md')

def show_page(did, p):
    display(IImage(str(PNG_DIR / f'{did}-p{int(p):02d}.png'), width=480))
    display(Markdown("**PyMuPDF**\n\n" + (PYM[did].get(str(p)) or '(없음)')))
    for m in RES:
        r = RES[m].get(did, {}).get(str(p))
        if r: display(Markdown(f"**{m}** ({r['sec']}s)\n\n" + (r['md'] or '(출력 없음)')))
# show_page(next(iter(DOCS)), 1)


In [ ]:
# 10b. HTML 리포트 — 페이지 그림(레이아웃 표 영역을 빨간 상자로) + PyMuPDF + 방법별 표를 한 파일에
import html as _html
from PIL import ImageDraw

def md_to_html(mdt):
    if not mdt: return '<p class="none">(출력 없음)</p>'
    out = []
    for block in re.split(r'\n\s*\n', mdt.strip()):
        rows = [ln for ln in block.splitlines() if '|' in ln and not re.fullmatch(r'[\s|:-]+', ln)]
        if not rows: out.append('<pre>' + _html.escape(block) + '</pre>'); continue
        out.append('<table>' + ''.join('<tr>' + ''.join(f'<td>{_html.escape(c.strip())}</td>' for c in r.strip().strip('|').split('|')) + '</tr>' for r in rows) + '</table>')
    return ''.join(out)

def page_img_html(did, i, width=720):
    img = Image.open(io.BytesIO(page_png(did, i, 110))).convert('RGB')
    try:                                   # 레이아웃 모델이 잡은 표 영역을 110 DPI 좌표로 환산해 그린다 (8번 셀 이후에만)
        _, boxes = table_boxes(page_png(did, i, 200)); k = 110 / 200
        d = ImageDraw.Draw(img)
        for x0, y0, x1, y1 in boxes: d.rectangle([x0 * k, y0 * k, x1 * k, y1 * k], outline='red', width=4)
    except Exception:
        boxes = []
    img.thumbnail((width, 10000)); buf = io.BytesIO(); img.save(buf, format='JPEG', quality=80)
    return f'<img src="data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}">', len(boxes)

ORDER = [m for m in ('A-page150', 'B-page200', 'C-crop200', 'D-pipeline', 'A15-page150', 'C15-crop200') if m in RES]
parts = ['<html><head><meta charset="utf-8"><style>body{font-family:sans-serif;font-size:13px} .page{display:flex;gap:16px;border-top:2px solid #333;padding:12px 0}'
         '.left{flex:0 0 720px} .right{flex:1;min-width:0} table{border-collapse:collapse;margin:4px 0;font-size:12px} td{border:1px solid #999;padding:2px 5px;vertical-align:top}'
         'details{margin:6px 0} summary{cursor:pointer;font-weight:bold} .none{color:#999} .meta{color:#555}</style></head><body>']
parts.append('<h1>PaddleOCR-VL 정확도 실험 — 페이지별 결과</h1><p>빨간 상자 = 레이아웃 모델(PP-DocLayout)이 표로 판정한 영역. C·D는 이 영역만 VLM에 넣는다.</p>')
for did, d in DOCS.items():
    for p, info in SEL[did]['pages'].items():
        img_tag, nbox = page_img_html(did, p)
        parts.append(f'<div class="page"><div class="left"><h3>{did} ({d["insurer"]}) p{p} — 원본 p{info["orig"]} [{info["kind"]}] · 레이아웃 표 {nbox}개</h3>{img_tag}</div><div class="right">')
        parts.append(f'<details><summary>PyMuPDF (텍스트 레이어)</summary>{md_to_html(PYM[did].get(p))}</details>')
        for m in ORDER:
            r = RES[m].get(did, {}).get(p)
            if not r: continue
            meta = f'{r["sec"]}s · 표 {r.get("n_tables")}' + (' · <b style="color:red">잘림</b>' if r.get('finish') == 'length' else '')
            parts.append(f'<details {"open" if m in ("A-page150", "C-crop200") else ""}><summary>{m} <span class="meta">({meta})</span></summary>{md_to_html(r["md"])}</details>')
        parts.append('</div></div>')
parts.append('</body></html>')
REPORT = OUT / 'report.html'; REPORT.write_text(''.join(parts), encoding='utf-8')
print("저장:", REPORT, f"({REPORT.stat().st_size // 1024}KB)")
from google.colab import files; files.download(str(REPORT))      # 브라우저로 내려받아 연다


In [ ]:
# 11. (선택) 이전 버전 1.5로 A와 C만 다시 — 1.6이 실제로 나아졌는지 같은 지표로 본다
RUN_15 = True
if RUN_15:
    serve_paddle('PaddlePaddle/PaddleOCR-VL-1.5', 'PaddleOCR-VL-1.5-0.9B')
    for did in DOCS:
        for i in SEL[did]['pages']:
            if not done('A15-page150', did, i):
                out, mdt, sec, ptok, fin = vlm(b64(page_png(did, i, 150)))
                put('A15-page150', did, i, {'raw': out, 'md': mdt, 'sec': sec, 'prompt_tokens': ptok, 'finish': fin, 'n_tables': 1 if mdt else 0})
            if not done('C15-crop200', did, i):
                t0 = time.perf_counter(); img, boxes = table_boxes(page_png(did, i, 200)); lay = round(time.perf_counter() - t0, 2)
                outs = [vlm(crop_b64(img, b)) for b in boxes]
                put('C15-crop200', did, i, {'raw': '\n\n'.join(o[0] for o in outs), 'md': '\n\n'.join(o[1] for o in outs if o[1]) or None,
                                             'sec': round(lay + sum(o[2] for o in outs), 2), 'prompt_tokens': max((o[3] or 0) for o in outs) if outs else None,
                                             'finish': 'length' if any(o[4] == 'length' for o in outs) else 'stop', 'n_boxes': len(boxes), 'n_tables': sum(1 for o in outs if o[1])})
        print("1.5", did, "완료")
    print("10번 셀을 다시 실행하면 1.5 행이 요약표에 더해진다")


## 읽는 법과 결정

| 결과 | 결정 |
|---|---|
| C ≈ D 이고 둘 다 A보다 셀일치가 높다 | 레이아웃 모델을 파이프라인에 넣는다. chunker 파드에서 CPU로 돌리고, 표 영역만 VLM에 보낸다. 본문오탐도 함께 사라진다 |
| A ≈ C | 페이지 전체 입력을 유지한다. 환경변수만 바꾸면 된다 |
| B > A | `VLM_DPI`를 200으로 올린다. 입력 토큰이 늘지 않는데 좋아졌다면 상한에 걸린 것이 아니라 렌더 품질 차이 |
| 1.6 ≈ 1.5 이하 | 드문 경우. 1.6이 기본이므로 1.5가 뚜렷이 낫다면 그때만 ID를 되돌린다 |
| D만 좋다 | 공식 파이프라인을 백엔드로 들인다. 의존이 가장 크지만 스캔 페이지 본문까지 해결된다 |

셀일치는 PyMuPDF가 읽는 괘선 표에서만 계산된다. gap 페이지는 compare.md에서 이미지와 직접 대조한다. 숫자와 질병코드 오독은 지표에 안 잡히므로 ruled 페이지 두어 장은 눈으로 본다.
